## _GOLD LAYER — Business Aggregations for Analytics_

In [0]:
from pyspark.sql.functions import (
    col, round, avg, sum as spark_sum, count,
    countDistinct, when, month, year, concat_ws
)

spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce_gold")

DataFrame[]

In [0]:
# ── Load Silver tables ───────────────────────────────────────
master      = spark.table("ecommerce_silver.master")
order_items = spark.table("ecommerce_silver.order_items")
orders      = spark.table("ecommerce_silver.orders")
reviews     = spark.table("ecommerce_silver.reviews")

print("✅ Silver tables loaded")

✅ Silver tables loaded


### GOLD 1 — Seller Performance Scorecard

In [0]:
seller_performance = (
    order_items
    .join(orders.select("order_id", "order_status", "is_late_delivery"),
          on="order_id", how="left")
    .join(reviews.select("order_id", "review_score"),
          on="order_id", how="left")
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        countDistinct("order_id").alias("total_orders"),
        round(spark_sum(col("price").cast("double")), 2).alias("total_revenue"),
        round(avg(col("price").cast("double")), 2).alias("avg_order_value"),
        round(avg("review_score"), 2).alias("avg_review_score"),
        round(
            (spark_sum("is_late_delivery") / count("order_id")) * 100, 2
        ).alias("late_delivery_pct")
    )
    .filter(col("total_orders") >= 5)  # exclude low-volume sellers
    .orderBy(col("total_revenue").desc())
)

seller_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_gold.seller_performance")

print(f"✅ seller_performance: {seller_performance.count():,} rows")

✅ seller_performance: 1,794 rows


### GOLD 2 — Monthly Revenue by Product Category

In [0]:
monthly_revenue = (
    order_items
    .join(orders.select("order_id", "order_purchase_timestamp", "order_status"),
          on="order_id", how="left")
    .filter(col("order_status") == "delivered")
    .withColumn("year",  year("order_purchase_timestamp"))
    .withColumn("month", month("order_purchase_timestamp"))
    .withColumn("year_month", concat_ws("-", col("year"), col("month")))
    .groupBy("year_month", "year", "month", "category_en")
    .agg(
        round(spark_sum(col("price").cast("double")), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        round(avg(col("price").cast("double")), 2).alias("avg_order_value")
    )
    .orderBy("year", "month")
)

monthly_revenue.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_gold.monthly_revenue")

print(f"✅ monthly_revenue: {monthly_revenue.count():,} rows")

✅ monthly_revenue: 1,264 rows


### GOLD 3 — State-level Customer & Revenue Distribution

In [0]:
state_distribution = (
    master
    .filter(col("order_status") == "delivered")
    .groupBy("customer_state")
    .agg(
        countDistinct("customer_id").alias("unique_customers"),
        countDistinct("order_id").alias("total_orders"),
        round(spark_sum(col("payment_value").cast("double")), 2).alias("total_revenue"),
        round(avg(col("payment_value").cast("double")), 2).alias("avg_order_value"),
        round(avg("review_score"), 2).alias("avg_review_score")
    )
    .orderBy(col("total_revenue").desc())
)

state_distribution.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_gold.state_distribution")

print(f"✅ state_distribution: {state_distribution.count():,} rows")

✅ state_distribution: 27 rows


### GOLD 4 — Late Delivery Heatmap by Seller City

In [0]:
late_delivery_heatmap = (
    order_items
    .join(orders.select("order_id", "is_late_delivery", "delivery_delay_days"),
          on="order_id", how="left")
    .groupBy("seller_city", "seller_state")
    .agg(
        count("order_id").alias("total_orders"),
        spark_sum("is_late_delivery").alias("late_orders"),
        round(
            (spark_sum("is_late_delivery") / count("order_id")) * 100, 2
        ).alias("late_delivery_pct"),
        round(avg("delivery_delay_days"), 1).alias("avg_delay_days")
    )
    .filter(col("total_orders") >= 10)
    .orderBy(col("late_delivery_pct").desc())
)

late_delivery_heatmap.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_gold.late_delivery_heatmap")

print(f"✅ late_delivery_heatmap: {late_delivery_heatmap.count():,} rows")

✅ late_delivery_heatmap: 374 rows


### GOLD 5 — Top Product Categories by Revenue

In [0]:
category_performance = (
    order_items
    .join(orders.select("order_id", "order_status"), on="order_id", how="left")
    .join(reviews.select("order_id", "review_score"), on="order_id", how="left")
    .filter(col("order_status") == "delivered")
    .groupBy("category_en")
    .agg(
        round(spark_sum(col("price").cast("double")), 2).alias("total_revenue"),
        countDistinct("order_id").alias("total_orders"),
        round(avg(col("price").cast("double")), 2).alias("avg_price"),
        round(avg("review_score"), 2).alias("avg_review_score")
    )
    .orderBy(col("total_revenue").desc())
)

category_performance.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce_gold.category_performance")

print(f"✅ category_performance: {category_performance.count():,} rows")

print("\n✅ GOLD LAYER COMPLETE")

✅ category_performance: 72 rows

✅ GOLD LAYER COMPLETE


In [0]:
spark.table("ecommerce_gold.vw_category_performance").printSchema()

root
 |-- category_en: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- avg_review_score: double (nullable = true)
 |-- revenue_share_pct: double (nullable = true)
 |-- revenue_rank: integer (nullable = false)



In [0]:
spark.table("ecommerce_gold.category_performance").printSchema()

root
 |-- category_en: string (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- total_orders: long (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- avg_review_score: double (nullable = true)



In [0]:
df = spark.table("ecommerce_gold.vw_category_performance")

(
    df.coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/Volumes/workspace/default/raw_data/tableau_exports/category_performance_v2_temp")
)

files = [f.path for f in dbutils.fs.ls("/Volumes/workspace/default/raw_data/tableau_exports/category_performance_v2_temp") if f.name.startswith("part-")]
dbutils.fs.cp(files[0], "/Volumes/workspace/default/raw_data/tableau_exports/category_performance_v2.csv")
dbutils.fs.rm("/Volumes/workspace/default/raw_data/tableau_exports/category_performance_v2_temp", recurse=True)
print("✅ Done")

✅ Done


In [0]:
df = spark.table("ecommerce_gold.vw_seller_performance")

(
    df.coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp")
)

files = [f.path for f in dbutils.fs.ls("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp") if f.name.startswith("part-")]
dbutils.fs.cp(files[0], "/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2.csv")
dbutils.fs.rm("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp", recurse=True)
print("✅ Done")

✅ Done


In [0]:
df = spark.table("ecommerce_gold.vw_seller_performance")

(
    df.coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp")
)

files = [f.path for f in dbutils.fs.ls("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp") if f.name.startswith("part-")]
dbutils.fs.cp(files[0], "/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2.csv")
dbutils.fs.rm("/Volumes/workspace/default/raw_data/tableau_exports/seller_performance_v2_temp", recurse=True)
print("✅ Done")

✅ Done


In [0]:
# Export state_distribution and late_delivery_heatmap views
exports = {
    "state_distribution_v2": "ecommerce_gold.vw_state_distribution",
    "late_delivery_v2": "ecommerce_gold.vw_late_delivery_heatmap",
}

base = "/Volumes/workspace/default/raw_data/tableau_exports/"

for filename, table in exports.items():
    df = spark.table(table)
    temp = f"{base}{filename}_temp"
    (
        df.coalesce(1)
        .write.option("header", "true")
        .mode("overwrite")
        .csv(temp)
    )
    files = [f.path for f in dbutils.fs.ls(temp) if f.name.startswith("part-")]
    dbutils.fs.cp(files[0], f"{base}{filename}.csv")
    dbutils.fs.rm(temp, recurse=True)
    print(f"✅ {filename}.csv exported")

✅ state_distribution_v2.csv exported
✅ late_delivery_v2.csv exported
